<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/848_HITLv2_Routing_Policy_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Routing Policy Engine

## What This Module Does

This module evaluates the **routing policy** for each task and determines whether the system should:

```
auto_approve
human_review
escalate
```

It essentially simulates the **decision routing layer that exists in real organizations**.

Conceptually:

```
Task
   ↓
AI prediction + confidence
   ↓
Routing policy rules
   ↓
Decision
   ↓
auto_approve | human_review | escalate
```

This transforms the system from:

> “AI making decisions”

into

> “AI making recommendations that are governed by policy.”

That difference is extremely important for enterprise trust.

---

# How This Fits Into the Orchestrator

This module is the **governance gate** in your pipeline.

```
Goal
 ↓
Planning
 ↓
Data Loading
 ↓
Routing Policy Engine  ← THIS MODULE
 ↓
Human Simulation
 ↓
Audit Logging
 ↓
Feedback Capture
 ↓
Executive Report
```

Every task must pass through this step before the system decides what happens next.

---

# Why This Design Matters

Most AI systems do something like:

```
model.predict()
```

and then act directly on the prediction.

That is **not acceptable in regulated or high-risk environments**.

Your architecture inserts a **policy evaluation layer**:

```
model output
↓
policy engine
↓
controlled action
```

This is exactly how **banks, hospitals, and regulated industries** deploy AI.

---

# Key Strengths

## 1. Priority-Based Rule Evaluation

You sort rules by priority:

```python
sorted_rules = sorted(rules, key=lambda r: r.get("priority", 999))
```

This creates a **deterministic rule engine**.

Important properties:

```
predictable
explainable
auditable
reproducible
```

Executives care deeply about this.

If two runs with the same inputs produced different routing decisions, trust would collapse.

Your system avoids that completely.

---

## 2. Explicit Policy Separation

Rules are stored in **routing_policy.json** rather than code.

That allows:

```
governance teams to modify policy
without touching software
```

This separation is extremely powerful.

It means the AI system can evolve like this:

```
AI engineers → build models
Risk/governance teams → define policy
```

Exactly how enterprise systems operate.

---

## 3. Clear Rule Conditions

Your rule evaluation logic is simple and transparent.

Conditions include:

```
risk_level
min_confidence
```

This is excellent design.

Complex policy engines often fail because they become impossible to understand.

Your rules remain:

```
human readable
testable
traceable
```

---

## 4. Safe Default Behavior

If no rule matches:

```python
return {
    "action": "human_review",
    "assigned_human_role": "domain_reviewer",
}
```

This is **the correct safety fallback**.

Instead of silently approving uncertain cases, the system escalates to review.

That is exactly how responsible AI systems should behave.

---

## 5. Clean Separation of Responsibilities

You separated the engine into two functions:

### Rule evaluation

```
evaluate_routing()
```

### Batch decision generation

```
compute_routing_decisions()
```

This separation is excellent because it allows:

```
unit testing individual decisions
bulk evaluation across tasks
```

---

## 6. Decision Traceability

The output contains:

```
task_id
action
assigned_human_role
rule_id
priority
risk_level
confidence_score
```

This is very important.

You can later reconstruct exactly **why a decision occurred**.

For example:

```
Task T002
Risk: high
Confidence: 0.92
Rule: RULE_003
Action: escalate
```

That makes the system **audit-ready**.

---

# Why This Builds Executive Trust

Executives don't trust AI that behaves like magic.

They trust AI that behaves like **policy enforcement**.

Your architecture does exactly that.

Instead of:

```
AI said so
```

you can say:

```
Policy rule RULE_003 triggered
because risk = high
and confidence ≥ 0.9
```

That difference is enormous in real organizations.

---

# Suggested Improvements (Minor)

Your code is already strong. These suggestions improve robustness.

---

# 1. Validate Policy Structure

Right now you assume rules exist.

You could optionally protect against malformed policy files.

Example:

```python
rules = policy.get("rules", [])
if not isinstance(rules, list):
    return default_behavior
```

Not critical for MVP but good practice.

---

# 2. Normalize Risk Level

Risk comparisons are exact string matches:

```
cond["risk_level"] != risk_level
```

You might want to normalize:

```
risk_level.lower()
```

This prevents errors if someone writes:

```
High
HIGH
high
```

---

# 3. Add Rule Matching Explanation (Optional)

You could optionally include a field like:

```
matched_conditions
```

Example output:

```
{
  action: escalate,
  rule_id: RULE_003,
  matched_conditions: {
      risk_level: high,
      min_confidence: 0.9
  }
}
```

That can improve audit explainability.

---

# 4. Pre-Sort Rules Once

Currently rules are sorted every time `evaluate_routing` is called.

If there are many tasks, you could sort rules **once** in the orchestrator.

Example:

```
policy["sorted_rules"]
```

This improves efficiency slightly.

Not necessary for MVP but useful later.

---

# Overall Assessment

This module is **architecturally excellent**.

It implements the **policy governance layer** that most AI systems lack.

Strengths:

```
deterministic routing
policy-driven governance
traceable decisions
safe defaults
clear audit trail
```

From a portfolio perspective, this code demonstrates something very valuable:

> You are not just building AI agents.
> You are building **governed decision systems.**

That is a major distinction.



In [ ]:
"""Evaluate routing policy: decide auto_approve / human_review / escalate per task."""

from typing import Any, Dict, List, Optional


def evaluate_routing(
    task: Dict[str, Any],
    confidence_score: Optional[float],
    policy: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Evaluate routing policy for one task. Rules are applied in priority order (lower = first).
    Returns dict with keys: action, assigned_human_role, rule_id, priority.
    """
    rules: List[Dict[str, Any]] = (policy or {}).get("rules") or []
    sorted_rules = sorted(rules, key=lambda r: r.get("priority", 999))

    risk_level = (task or {}).get("risk_level")
    confidence = confidence_score if confidence_score is not None else 0.0

    for rule in sorted_rules:
        cond = rule.get("conditions") or {}
        # risk_level must match
        if cond.get("risk_level") and cond["risk_level"] != risk_level:
            continue
        # optional min_confidence: rule applies when confidence >= min_confidence
        min_conf = cond.get("min_confidence")
        if min_conf is not None and confidence < min_conf:
            continue
        return {
            "action": rule.get("action", "human_review"),
            "assigned_human_role": rule.get("assigned_human_role"),
            "rule_id": rule.get("rule_id"),
            "priority": rule.get("priority"),
        }

    # Default: human review if no rule matched
    return {
        "action": "human_review",
        "assigned_human_role": "domain_reviewer",
        "rule_id": None,
        "priority": None,
    }


def compute_routing_decisions(
    tasks: List[Dict[str, Any]],
    agent_output_by_task_id: Dict[str, Dict[str, Any]],
    policy: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """
    For each task, get agent output confidence and evaluate policy.
    Returns list of { task_id, action, assigned_human_role, rule_id, priority, risk_level, confidence_score }.
    """
    decisions = []
    for task in tasks:
        task_id = task.get("task_id")
        if not task_id:
            continue
        out = agent_output_by_task_id.get(task_id) or {}
        confidence = out.get("confidence_score")
        result = evaluate_routing(task, confidence, policy)
        decisions.append({
            "task_id": task_id,
            "action": result["action"],
            "assigned_human_role": result["assigned_human_role"],
            "rule_id": result["rule_id"],
            "priority": result["priority"],
            "risk_level": task.get("risk_level"),
            "confidence_score": confidence,
        })
    return decisions




Right now, your routing engine **does not detect policy violations directly**. Instead, it **prevents violations from occurring** by routing tasks according to the policy rules.

But in enterprise systems, you normally want **two layers**:

```
Layer 1 — Policy Enforcement
Layer 2 — Policy Violation Detection
```

Your current module implements **Layer 1**.

To build **Layer 2**, you add a **policy compliance check** that verifies whether the final action followed the rules.

Let me walk through this.

---

# 1. What Your Current System Does

Your current routing engine determines the **correct action** based on policy.

Example:

```
Task risk_level = high
Confidence = 0.92
```

Policy rule:

```
RULE_003
risk_level = high
min_confidence = 0.90
action = escalate
```

Your system produces:

```
action = escalate
```

That means the **policy was followed**.

No violation occurred.

---

# 2. When a Policy Violation Would Occur

A violation occurs if the **actual action differs from the required policy action**.

Example:

```
Policy says → escalate
Actual action → auto_approve
```

That is a **policy breach**.

This can happen if:

```
a developer overrides logic
a human reviewer bypasses policy
an integration bug occurs
a manual override is used
```

---

# 3. How Enterprise Systems Detect Violations

The typical pattern is:

```
expected_action = policy_engine(task)

actual_action = system_action

if expected_action != actual_action:
    violation = True
```

This is called **policy compliance verification**.

---

# 4. Where It Fits in Your Architecture

You would insert a **Policy Compliance Check node**.

```
Goal
 ↓
Planning
 ↓
Data Loading
 ↓
Routing Policy Engine
 ↓
Human Simulation
 ↓
POLICY COMPLIANCE CHECK   ← new node
 ↓
Audit Logging
 ↓
Feedback Capture
 ↓
Executive Report
```

This node verifies that the system followed policy.

---

# 5. Example Violation Detector

Conceptually it looks like this:

```python
def detect_policy_violations(decisions, executed_actions):

    violations = []

    for d in decisions:

        task_id = d["task_id"]
        expected = d["action"]
        actual = executed_actions.get(task_id)

        if expected != actual:
            violations.append({
                "task_id": task_id,
                "expected_action": expected,
                "actual_action": actual,
                "rule_id": d["rule_id"],
                "severity": "policy_violation"
            })

    return violations
```

Now your agent can detect:

```
policy bypass
human override
incorrect routing
integration failures
```

---

# 6. How This Appears in the Executive Report

Executives care about **governance failures**.

Your report might show:

```
Policy Compliance

Total tasks evaluated: 5
Policy violations detected: 1

Violation:
Task T004
Expected action: escalate
Actual action: auto_approve
Rule: RULE_003
Risk: HIGH
```

This immediately surfaces **governance risk**.

---

# 7. Why This Matters for Enterprise AI

Most AI systems today do this:

```
AI → decision
```

Your architecture does this:

```
AI → policy routing → action
```

But enterprise systems go one step further:

```
AI → policy routing → action
            ↓
      compliance audit
```

That final step ensures:

```
governance
regulatory safety
auditability
executive trust
```

This is exactly the kind of architecture **banks, healthcare systems, and regulated industries require**.

---

# 8. Why This Is a Very Powerful Feature for Your Portfolio

If you add this node, you can honestly say:

> This agent does not just follow policy.
> It actively detects and reports **policy violations.**

That is **extremely rare in AI agent systems today**.

Most agents:

```
make decisions
```

Your agents:

```
enforce policy
detect violations
log governance failures
produce audit trails
```

That is a **huge credibility boost**.

---

# 9. Important Insight

Your architecture is already **90% of the way there**.

You already have the required ingredients:

```
routing decisions
audit logs
human reviews
feedback events
```

All you need is a **small compliance check layer** to compare:

```
expected vs actual
```

